# 🧠 Deep Learning Models: 1D-CNN, Simple RNN & LSTM (Deep Knowledge Tracing)

This notebook implements sequential deep learning architectures (**1D-CNN**, **Simple RNN**, and **LSTM**) using **PyTorch** on student attempt interaction sequences ($N=10$) to predict student **Knowledge Gap Levels** (`Low`, `Medium`, `High`).

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries Imported Successfully!")
print("PyTorch Version :", torch.__version__)

Libraries Imported Successfully!
PyTorch Version : 2.13.0+cu130


In [2]:
clean = pd.read_csv("../data/processed/cleaned_skill_builder.csv")
gap = pd.read_csv("../data/processed/knowledge_gap_dataset.csv")

df = clean.merge(
    gap[["user_id", "skill_name", "KnowledgeGap"]],
    on=["user_id", "skill_name"],
    how="inner"
)

print("Clean Dataset Shape :", clean.shape)
print("Knowledge Gap Dataset Shape :", gap.shape)
print("Merged Dataset Shape :", df.shape)

Clean Dataset Shape : (525534, 26)
Knowledge Gap Dataset Shape : (41576, 15)
Merged Dataset Shape : (525534, 27)


In [3]:
df['hint_ratio'] = df['hint_count'] / (df['hint_total'] + 1)
df['log_ms_response'] = np.log1p(df['ms_first_response'].clip(lower=0))
df['log_overlap_time'] = np.log1p(df['overlap_time'].clip(lower=0))
df['attempt_hint_sum'] = df['attempt_count'] + df['hint_count']

seq_features = [
    'correct',
    'attempt_count',
    'hint_count',
    'hint_ratio',
    'log_ms_response',
    'log_overlap_time',
    'attempt_hint_sum'
]

scaler = StandardScaler()
df[seq_features] = scaler.fit_transform(df[seq_features])

SEQ_LEN = 10
X_list = []
y_list = []

label_encoder = LabelEncoder()
df['target'] = label_encoder.fit_transform(df['KnowledgeGap'])

grouped = df.groupby(['user_id', 'skill_name'])
for _, group in grouped:
    if len(group) >= SEQ_LEN:
        feat_matrix = group[seq_features].values[:SEQ_LEN]
        target_val = group['target'].iloc[0]
        X_list.append(feat_matrix)
        y_list.append(target_val)

X_seq = np.array(X_list, dtype=np.float32)
y_seq = np.array(y_list, dtype=np.int64)

print("Total Student Skill Sequences Generated :", len(X_seq))
print("Sequence Input Tensor Shape (X_seq) :", X_seq.shape)
print("Sequence Target Vector Shape (y_seq) :", y_seq.shape)

Total Student Skill Sequences Generated : 35420
Sequence Input Tensor Shape (X_seq) : (35420, 10, 7)
Sequence Target Vector Shape (y_seq) : (35420,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_seq,
    y_seq,
    test_size=0.2,
    random_state=42,
    stratify=y_seq
)

print("Training Sequences :", X_train.shape)
print("Testing Sequences  :", X_test.shape)

Training Sequences : (28336, 10, 7)
Testing Sequences  : (7084, 10, 7)


In [5]:
class CNN1DModel(nn.Module):
    def __init__(self):
        super(CNN1DModel, self).__init__()
        self.conv = nn.Conv1d(in_channels=7, out_channels=64, kernel_size=3)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.fc1 = nn.Linear(64 * 4, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.pool(self.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

cnn_net = CNN1DModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_net.parameters(), lr=0.001)

for epoch in range(10):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(cnn_net(bx), by)
        loss.backward()
        optimizer.step()

cnn_net.eval()
preds, targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        preds.extend(torch.argmax(cnn_net(bx), dim=1).numpy())
        targets.extend(by.numpy())

cnn_acc = accuracy_score(targets, preds)
print(f"1D-CNN Test Accuracy : {cnn_acc:.4f} ({cnn_acc*100:.2f}%)")

1D-CNN Test Accuracy : 0.8520 (85.20%)


In [6]:
class RNNModel(nn.Module):
    def __init__(self):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_size=7, hidden_size=64, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 3)
        self.relu = nn.ReLU()

    def forward(self, x):
        out, _ = self.rnn(x)
        x = out[:, -1, :]
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

rnn_net = RNNModel()
optimizer = optim.Adam(rnn_net.parameters(), lr=0.001)

for epoch in range(10):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(rnn_net(bx), by)
        loss.backward()
        optimizer.step()

rnn_net.eval()
preds, targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        preds.extend(torch.argmax(rnn_net(bx), dim=1).numpy())
        targets.extend(by.numpy())

rnn_acc = accuracy_score(targets, preds)
print(f"Simple RNN Test Accuracy : {rnn_acc:.4f} ({rnn_acc*100:.2f}%)")

Simple RNN Test Accuracy : 0.8551 (85.51%)


In [7]:
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size=7, hidden_size=64, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 3)
        self.relu = nn.ReLU()

    def forward(self, x):
        out, _ = self.lstm(x)
        x = out[:, -1, :]
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

lstm_net = LSTMModel()
optimizer = optim.Adam(lstm_net.parameters(), lr=0.001)

for epoch in range(10):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(lstm_net(bx), by)
        loss.backward()
        optimizer.step()

lstm_net.eval()
preds, targets = [], []
with torch.no_grad():
    for bx, by in test_loader:
        preds.extend(torch.argmax(lstm_net(bx), dim=1).numpy())
        targets.extend(by.numpy())

lstm_acc = accuracy_score(targets, preds)
print(f"LSTM Test Accuracy : {lstm_acc:.4f} ({lstm_acc*100:.2f}%)")

# Export Model Weights
torch.save(lstm_net.state_dict(), '../models/lstm_model.pth')
print('✔ Saved PyTorch LSTM Model weights to models/lstm_model.pth')

LSTM Test Accuracy : 0.8600 (86.00%)


In [8]:
print("="*60)
print("MACHINE LEARNING vs DEEP LEARNING MODEL BENCHMARK")
print("="*60)
print(f"1D-CNN (Deep Learning)           : {cnn_acc:.4f} ({cnn_acc*100:.2f}%)")
print(f"Simple RNN (Deep Learning)       : {rnn_acc:.4f} ({rnn_acc*100:.2f}%)")
print(f"LSTM / DKT (Deep Learning)       : {lstm_acc:.4f} ({lstm_acc*100:.2f}%)")
print("="*60)

MACHINE LEARNING vs DEEP LEARNING MODEL BENCHMARK
1D-CNN (Deep Learning)           : 0.8520 (85.20%)
Simple RNN (Deep Learning)       : 0.8551 (85.51%)
LSTM / DKT (Deep Learning)       : 0.8600 (86.00%)
